# Logistic Regression — Fraud Detection Baseline

**Goal:** establish the must-have baseline before reaching for anything fancier.

We'll cover: preprocessing pipeline, class weighting for imbalance, PR-AUC vs ROC-AUC, and coefficient interpretation.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix
)

SEED = 42
df = pd.read_parquet(Path('..') / 'data' / 'transactions.parquet')
print(df.shape)
df.head()

## Quick look at class balance

Fraud datasets are extremely imbalanced. Accuracy is useless here — a model that predicts 'never fraud' is ~98% accurate and 0% useful.

In [ ]:
print(df['is_fraud'].value_counts(normalize=True).rename('fraction'))
print(f"\nBaseline (predict 0): accuracy = {1 - df['is_fraud'].mean():.4f}")

## Build a preprocessing + model pipeline

Why a `Pipeline`?
- Prevents leakage: scalers/encoders fit on the train fold only.
- One object to serialize for production.
- Clean cross-validation.

In [ ]:
numeric_cols = ['account_age_days', 'txn_amount', 'velocity_1h',
                'device_entropy', 'email_risk', 'ip_country_mismatch',
                'noise_1', 'noise_2', 'noise_3']
categorical_cols = ['device_type', 'country']

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
    ('cat', categorical_pipe, categorical_cols),
])

model = Pipeline([
    ('prep', preprocessor),
    ('lr', LogisticRegression(
        max_iter=1000,
        class_weight='balanced',   # <-- key for imbalance
        C=1.0,
        random_state=SEED,
    )),
])
model

## Stratified train/test split

Stratify on the label so train and test have the same fraud rate. In production you'd split by **time** to avoid temporal leakage — see module 04.

In [ ]:
X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
print(f"train fraud rate: {y_train.mean():.4f}  |  test fraud rate: {y_test.mean():.4f}")

In [ ]:
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred, digits=4))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"PR-AUC : {average_precision_score(y_test, y_proba):.4f}  <-- the metric that matters here")

## Choosing an operating threshold

In fraud, you rarely deploy at 0.5. You pick a threshold that hits a target — e.g., 'review the top 1% of traffic' or 'keep false-positive rate below 0.5%'.

In [ ]:
# Threshold that flags the top 2% of traffic (matches the fraud base rate).
k = int(0.02 * len(y_proba))
threshold = np.sort(y_proba)[-k]
y_pred_k = (y_proba >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_k).ravel()
print(f"Threshold @ top 2%: {threshold:.4f}")
print(f"Precision: {tp / (tp + fp):.4f}")
print(f"Recall   : {tp / (tp + fn):.4f}")

## Coefficient interpretation

A standardized linear model lets us read coefficients as **standardized log-odds contributions**. Big positive = pushes prediction toward fraud.

In [ ]:
feature_names = model.named_steps['prep'].get_feature_names_out()
coefs = model.named_steps['lr'].coef_[0]
imp = (pd.DataFrame({'feature': feature_names, 'coef': coefs})
         .assign(abs_coef=lambda d: d['coef'].abs())
         .sort_values('abs_coef', ascending=False)
         .drop(columns='abs_coef')
         .head(12))
imp

### Sanity check

If you generated the data yourself, you know the **real** signals are `account_age_days`, `txn_amount`, `velocity_1h`, `device_entropy`, `email_risk`, `ip_country_mismatch`, and country. The `noise_*` features should sit near zero. If they don't, you're overfitting — increase regularization or add more data.

**Takeaway:** start every problem with a linear baseline. If a fancier model can't beat it by a meaningful margin on PR-AUC, the fancier model isn't earning its complexity.